<a href="https://colab.research.google.com/github/Clei21/Projeto2/blob/main/tp2_experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Trade-off entre Especialização e Generalização em LLMs via Fine-Tuning


## 1. Montar o Google Drive e preparar o ambiente

In [15]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/tp2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Backup em:', DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backup em: /content/drive/MyDrive/tp2


In [ ]:
%cd /content
!rm -rf Projeto2
!git clone https://github.com/Clei21/Projeto2.git
%cd Projeto2

# garante a versão compatível do protobuf (correção do conflito com o deepeval)
!sed -i 's/protobuf==5.28.3/protobuf==4.25.5/' requirements.txt
!pip install -q -r requirements.txt
print('Instalação concluída.')

/content
Cloning into 'Projeto2'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 83 (delta 9), reused 73 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 67.81 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/Projeto2


> Se ao final aparecer um aviso pedindo para **reiniciar a sessão** (restart session),
> reinicie e execute novamente apenas as células da seção 1 — a reinstalação será quase instantânea.

## 2. Download do Spider



In [ ]:
SPIDER_GDRIVE_ID = '1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J'

!pip install -q gdown
!gdown {SPIDER_GDRIVE_ID} -O spider.zip
!unzip -q -o spider.zip
!ls spider

A listagem acima deve conter `train_spider.json`, `dev.json`, `tables.json` e `database/`.
Se a pasta extraída tiver outro nome (e.g. `spider_data`), renomeie com `!mv spider_data spider`.

## 3. Pré-processamento dos dados (Spider + suíte MMLU)

In [ ]:
!python scripts/prepare_spider.py --spider_dir spider --out_dir data
!python scripts/build_mmlu_suite.py --out data/mmlu_suite.json
!cp -r data {DRIVE_DIR}/

## 4. Teste rápido do pipeline (~15 min)

Valida geração + métrica com apenas 20 exemplos antes das execuções longas.
Deve terminar imprimindo `Execution Accuracy: 0.xxxx`.

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/teste.jsonl --limit 20

!PREDICTIONS=results/teste.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s

## 5. Baseline — Fase 2 (~2–4 h)

Avaliação do modelo base (sem treino) no dev split completo, com o prompt few-shot fixo
e decodificação greedy.

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_baseline.jsonl

!PREDICTIONS=results/predictions_baseline.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

## 6. Fine-tuning — Fase 3 (~4–6 h por configuração)

Duas configurações obrigatórias: A (lr = 2e-4) e B (lr = 2e-5). Pelo limite de sessão do
Colab gratuito, o recomendado é **uma configuração por sessão** (e.g. A hoje, B amanhã).
O adaptador é copiado para o Drive ao final.

In [ ]:
!python scripts/train_lora.py --config configs/lora_a.yaml
!cp -r outputs {DRIVE_DIR}/

In [ ]:
!python scripts/train_lora.py --config configs/lora_b.yaml
!cp -r outputs {DRIVE_DIR}/

### Recuperar adaptadores do Drive (use apenas se a sessão caiu)

Se você está em uma sessão nova e os treinos já foram feitos antes, execute a célula abaixo
para restaurar os adaptadores sem treinar de novo.

In [ ]:
!cp -r {DRIVE_DIR}/outputs . 2>/dev/null || echo 'nenhum adaptador no Drive ainda'
!cp -r {DRIVE_DIR}/results . 2>/dev/null || true
!cp -r {DRIVE_DIR}/data . 2>/dev/null || true
!ls outputs 2>/dev/null || true

## 7. Avaliação dos modelos fine-tuned — Fase 4 (~2–4 h por modelo)

Mesmo procedimento do baseline, apenas com `--adapter`.

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct --adapter outputs/lora_a \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_lora_a.jsonl

!PREDICTIONS=results/predictions_lora_a.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct --adapter outputs/lora_b \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_lora_b.jsonl

!PREDICTIONS=results/predictions_lora_b.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

## 8. Regressão de capacidade no MMLU — Fase 5 (~20 min por modelo)

Avaliação 5-shot nas 150 questões, para o modelo base e os dois fine-tuned.

In [ ]:
!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --suite data/mmlu_suite.json --out results/mmlu_baseline.json

!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --adapter outputs/lora_a --suite data/mmlu_suite.json --out results/mmlu_lora_a.json

!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --adapter outputs/lora_b --suite data/mmlu_suite.json --out results/mmlu_lora_b.json

!cp -r results {DRIVE_DIR}/

## 9. Consolidação do trade-off (instantâneo)

In [ ]:
!python scripts/analyze_tradeoff.py \
  --spider_baseline results/eval_predictions_baseline.json \
  --spider_finetuned results/eval_predictions_lora_a.json \
  --mmlu_baseline results/mmlu_baseline.json \
  --mmlu_finetuned results/mmlu_lora_a.json \
  --out results/tradeoff_lora_a.json

In [ ]:
!python scripts/analyze_tradeoff.py \
  --spider_baseline results/eval_predictions_baseline.json \
  --spider_finetuned results/eval_predictions_lora_b.json \
  --mmlu_baseline results/mmlu_baseline.json \
  --mmlu_finetuned results/mmlu_lora_b.json \
  --out results/tradeoff_lora_b.json

!cp -r results {DRIVE_DIR}/

## 10. Próximos passos (fora do Colab)

1. Os números para o relatório estão em `results/` (e no Drive, em `tp2/results/`):
   `eval_predictions_*.json` (Execution Accuracy), `mmlu_*.json` e `tradeoff_*.json`.
2. Para a análise de erros, abra `eval_predictions_lora_a.json` e procure entradas com
   `"score": 0.0`; compare a SQL gerada (em `predictions_lora_a.jsonl`) com a gold.
3. Preencha os campos `[PREENCHER]` de `report/relatorio.tex` e compile no Overleaf.
4. Suba o notebook executado e os resultados para o repositório do GitHub.